# Ejemplo calcular TF-IDF y similitud coseno en CiperChile

**Curso**: INF3841 - Recuperación de Información  
**Profesor**: Juan Manuel Barrios  
**Fecha**: 21 de septiembre de 2026

En ese ejemplo calcularemos la similitud global entre documentos de texto.

Para cada archivo de texto se calcula un descriptor global (llamado TF-IDF).

Esos descriptores se comparan y para cada documento se obtendrá el documento más parecido dentro del mismo conjunto.

```
pip install scikit-learn pandas
```

Se usará el dataset `dataset_ciperchile.20-09-2026.zip` con artículos del sitio https://www.ciperchile.cl/  
Ver la descripción en su archivo `Readme.txt`.  
Dataset publicado solo para fines académicos.  
El derecho de autor de todos los textos pertenece a CIPER Chile. 

In [ ]:
import zipfile
import time
import numpy
import os


class File:
    def __init__(self):
        self.filename = ""
        self.titulo = ""
        self.autor = ""
        self.fecha = ""
        self.tipo = ""
        self.url = ""
        self.full_text = ""


class Dataset:
    def __init__(self, file_zip):
        self.dataset_files = []
        header = dict()
        print("leyendo {}...".format(file_zip))
        t1 = time.time()
        with zipfile.ZipFile(file_zip, mode="r") as zip:
            with zip.open("dataset.txt", mode="r") as f:
                for line in f:
                    fields = line.decode('utf-8').strip().split("\t")
                    # leer los nombres de la primera fila
                    if len(header) == 0:
                        header = dict((c, i) for i, c in enumerate(fields))
                        continue
                    file = File()
                    file.filename = fields[header["Filename"]]
                    file.titulo = fields[header["Título"]]
                    file.autor = fields[header["Autor"]]
                    file.fecha = fields[header["Fecha"]]
                    file.tipo = fields[header["Tipo_documento"]]
                    file.url = fields[header["URL"]]
                    self.dataset_files.append(file)
            print("leyendo {} archivos en {} ...".format(len(self.dataset_files), file_zip))
            for file in self.dataset_files:
                with zip.open("dataset/" + file.filename) as f:
                    full_text = f.read().decode('utf-8')
                    # quitar la primera linea (URL)
                    file.full_text = full_text[full_text.find("\n") + 1 :]
        t2 = time.time()
        print("dataset cargado en {:.1f} segundos".format(t2 - t1))


dataset = Dataset("dataset_ciperchile.20-09-2026.zip")

## Seleccionar los documentos a usar

A veces es muy lento usar todos los documentos del dataset o puede faltar memoria.

Cambiar el número de documentos para reducir la cantidad de archivos a procesar.

In [ ]:
def fulltexts_documentos(dataset, num_documentos):
    textos = []
    documentos = []
    contador = 0
    largo_total = 0
    for doc in dataset.dataset_files[0:num_documentos]:
        textos.append(doc.full_text)
        documentos.append(doc)
        contador += 1
        largo_total += len(doc.full_text)
    print("Se leen {} documentos con largo total de {:.1f} MB".format(contador, largo_total / 1000 / 1000))
    return textos, documentos


# cantidad de documentos a leer, se puede reducir si es muy lento
num_documentos = len(dataset.dataset_files)     # un numero entre 1 y len(dataset.dataset_files)

# un array con los textos para crear el vocabulario y descriptores
textos, documentos = fulltexts_documentos(dataset, num_documentos)

## Calcular vocabulario

Se usa **TfidfVectorizer** para calcular el vocabulario y crear descriptores.  
Ver la documentación en: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# calcular el vocabulario: total de palabras a representar
t0 = time.time()
vectorizer = TfidfVectorizer(
    analyzer="word", # definición de token o gram, por defecto "word" (dividir en palabras)
    lowercase=True,  # convertir a minusculas, por defecto es True
    strip_accents="unicode",  # por defecto es None. Usar "unicode" para eliminar acentos.
    sublinear_tf=True,  # por defecto es False. Usar True para 1+log(freq)
    use_idf=True,       # por defecto es True.
    norm="l2",          # por defecto es "l2", crea vectores unitarios y el coseno es solo la multiplicacion de valores.
    ngram_range=(1,1),  # rango de ngramas a usar por ej: (1,1) o (1,2) o (1,3). Por defecto es (1,1).
    max_df=1.0,  # Descartar palabras que aparecen en más de max_df documentos. Probar con 0.9.
                 #    -Si es int   -> cantidad de documentos
                 #    -Si es float -> porcentaje del total de documentos
                 #    -Notar 1 que es distinto a 1.0 (solo 1 documento vs el 100% de los documentos)
    min_df=0.0,  # Descartar palabras que aparecen en menos de min_df documentos (int o float). Probar con 0.1.
)
vectorizer.fit(textos)
t1 = time.time()

print("tiempo para crear vocabulario: {:.1f} segs".format(t1 - t0))
print("vocabulario de {} palabras".format(len(vectorizer.vocabulary_)))
print("primeras 10 palabras: ", list(vectorizer.vocabulary_.keys())[0:10])
print("primeras 10 palabras ordenadas: ", list(sorted(vectorizer.vocabulary_.keys())[0:10]))


## Calcular descriptores

Se calcula el descriptor tf-idf para cada documento.

Notar que `TfidfVectorizer` entrega los descriptores como una matriz de tipo `scipy.sparse._csr.csr_matrix`. Esto implica que ocupa poco espacio en memoria porque solo se guardan los valores distintos de cero.


In [ ]:
# transform() entrega el descriptor tf-idf (un vector sparse con los pesos de las palabras)
t0 = time.time()
descriptores = vectorizer.transform(textos)
t1 = time.time()

print("tiempo descriptores: {:.1f} segs".format(t1 - t0))
print("descriptores es una matriz de {}".format(descriptores.shape))

# verificar que los descriptores es una matriz sparse
print("descriptores es tipo {}".format(type(descriptores)))

# Calcular similares (opción 1: multiplicar matrices)

Calcularemos la "autosimilitud" que consiste en comparar un dataset consigo mismo. Como los descriptores están normalizados L2, al multiplicar la matriz de descriptores con su transpuesta es lo mismo que calcular la similitud coseno entre todos los descriptores.

La matriz de autosimilitud  siempre tendrá 1 en la diagonal porque representa la similitud de un descriptor consigo mismo.

Notar que la matriz es tipo `sparse`, por lo que para poder multiplicar con ella será necesario convertirla en matriz densa, llamando a `toarray()`. Dependiendo de la cantidad de términos del vocabulario esto podría fallar por memoria.

La multiplicación de matrices debiera tomar algunos segundos en completar, un minuto como máximo (depende de la performance del procesador).

In [ ]:
# usar la funcion toarray() para convertir los descriptores a una matriz densa
# multiplicar las matrices con matmul() para calcular la similitud coseno

t0 = time.time()
descriptores_denso = descriptores.toarray()
auto_similitud = numpy.matmul(descriptores_denso, descriptores_denso.T)
t1 = time.time()

print("Tiempo comparación todos contra todos: {:.1f} segs".format(t1 - t0))

# se imprime la matriz de autosimilitud
auto_similitud

## Imprimir el resultado de similitud coseno

Para cada fila se imprime su más cercano (mayor similitud). Para evitar el valor 1 de la diagonal se escribe 0 en la diagonal.

In [ ]:
import pandas


def imprimir_similares(matriz, documentos, min_score):
    # obtener el maximo por fila
    # primero llenar la diagonal con ceros para que el mas parecido no sea si mismo
    numpy.fill_diagonal(matriz, 0)
    # recorrer las filas de la matriz
    filas = []
    for i in range(matriz.shape[0]):
        # obtener la posicion del maximo por fila
        posicion_mayor = numpy.argmax(matriz[i])
        # nombres de archivos
        documento = documentos[i]
        documento_mas_similar = documentos[posicion_mayor]
        similitud = matriz[i, posicion_mayor]
        # imprimir documentos con score mayor al mínimo pedido
        if similitud >= min_score:
            fila = {
                "query": documento.filename,
                "documento": documento_mas_similar.filename,
                "similitud coseno": similitud,
            }
            filas.append(fila)
    print("{} documentos con similitud mayor a {}:".format(len(filas), min_score))
    print()
    df = pandas.DataFrame(filas)
    print(df.to_string(index=False, justify="center"))


imprimir_similares(auto_similitud, documentos, min_score=0.0)

## Revisar algunos documentos parecidos

Se muestra un documento al azar y su más parecido, para verificar si son similares o no.

In [ ]:
import random


def imprimir_documento(documentos, similitudes, filename):
    posicion = 0
    documento = None
    for doc in documentos:
        if doc.filename == filename:
            documento = documentos[posicion]
            break
        posicion += 1
    if documento is None:
        print("no encuentro documento " + filename)
        return
    # obtener el documento y el más parecido
    posicion_mayor = numpy.argmax(similitudes[posicion])
    documento_mas_similar = documentos[posicion_mayor]
    similitud = similitudes[posicion, posicion_mayor]
    # mostrar el inicio de cada coumento
    max_length = 800
    texto_documento = documento.full_text
    texto_similar = documento_mas_similar.full_text
    if len(texto_documento) > max_length:
        texto_documento = texto_documento[0 : max_length - 6] + " (...)"
    if len(texto_similar) > max_length:
        texto_similar = texto_similar[0 : max_length - 6] + " (...)"
    print("Comparando {} y {}".format(documento.filename, documento_mas_similar.filename))
    print("similitud={:.3f}".format(similitud))
    print("--------------------------------------")
    print("  ", documento.filename, " -> ", documento.url)
    print(texto_documento)
    print("--------------------------------------")
    print("--------------------------------------")
    print("  ", documento_mas_similar.filename, " -> ", documento_mas_similar.url)
    print(texto_similar)
    print("--------------------------------------")


# mostrar un documento al azar
documento_azar = documentos[random.randint(0, len(documentos)-1)]
imprimir_documento(documentos, auto_similitud, documento_azar.filename)

##### __Preguntas:__
  * En su opinión, ¿son parecidos los dos documentos mostrados anteriormente? (pruebe varias veces)
  * ¿Tiene que ver el nivel parecido con el valor de similitud? (revisar casos con similitud 0.1 vs similitud 0.5)

# Calcular similares (opción 2: usar cdist)

En vez de multiplicar matrices probaremos la función `cdist()` que ya hemos usado en ejemplos anteriores. Para comparar descriptores configurar la distancia `cosine`, que es igual a (1 - similitud coseno).  

Ver definición de métrica 6 en: https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.cdist.html

**Notar la diferencia de tiempo comparado con la multiplicación de matrices!!**   
Usando `cdist()` para comparar descriptores **demora cerca de una hora**, mientras que la multiplicación de matrices toma algunos segundos. En ambos casos se obtiene exactamente el mismo resultado.

¿A qué se deberá la diferencia en tiempo?   
La diferencia fundamental entre multiplicar matrices vs `cdist()` es que este último debe normalizar (innecesariamente) los vectores al calcular cada distancia.

In [ ]:
from scipy.spatial import distance

t0 = time.time()
descriptores_denso = descriptores.toarray()
distancias = distance.cdist(descriptores_denso, descriptores_denso, metric="cosine")
t1 = time.time()

print(
    "tiempo comparación usando cdist: {:.1f} segs ({:.1f} mins)".format(
        t1 - t0, (t1 - t0) / 60
    )
)

In [ ]:
import pandas


def evaluar_mas_parecido(matriz_distancias, matriz_similitudes, documentos):
    # completar la diagonal con un valor muy grande para el mas cercano no sea si mismo
    numpy.fill_diagonal(matriz_distancias, numpy.inf)
    # completo la diagonal con cero para que el mas parecido no sea si mismo
    numpy.fill_diagonal(matriz_similitudes, 0)
    # obtener la posicion del mas cercano por fila
    posicion_cercano = numpy.argmin(matriz_distancias, axis=1)
    # posicion del mas similar por fila
    posicion_mas_similar = numpy.argmax(matriz_similitudes, axis=1)
    # imprimir los documentos mas cercanos y su similitud
    filas = []
    total = matriz_distancias.shape[0]
    total_iguales = 0
    for i in range(total):
        documento = documentos[i]
        documento_cercano = documentos[posicion_cercano[i]]
        distancia = matriz_distancias[i, posicion_cercano[i]]
        documento_mas_similar = documentos[posicion_mas_similar[i]]
        resultado = ""
        if documento_cercano == documento_mas_similar:
            resultado = "¡igual! :-)"
            total_iguales += 1
        else:
            resultado = "¡mal! era {}".format(documento_mas_similar.filename)
        fila = {
            "query": documento.filename,
            "documento": documento_cercano.filename,
            "distancia": distancia,
            "resultado": resultado,
        }
        filas.append(fila)
    print(
        "resultado igual a multiplicacion matrices: {} / {} = {:.1f}%".format(
            total_iguales, total, total_iguales / total * 100
        )
    )
    print()
    df = pandas.DataFrame(filas)
    print(df.to_string(index=False, justify="center"))


evaluar_mas_parecido(distancias, auto_similitud, documentos)

Es interesante cambiar en la función `distance.cdist()` el parámetro `metric="cosine"` por alguna otra función de distancia y ver cómo cambia el resultado.

En particular, explique por qué no cambia el resultado si se usa `metric="euclidean"` o `metric="sqeuclidean"`.

# Ejercicio Propuesto

Buscar documentos similares para textos de consulta libres.

In [ ]:
# Calcular el documento más similar para los siguientes documentos de consulta.
textos_q = ["elecciones de alcalde", "estudiantes de universidad"]

# 1.calcular la matriz de descriptores para textos_q (usar el vocabulario ya calculado)
# 2.multiplicar los descriptores para obtener la similitud coseno
# 2.buscar para cada descriptor de textos_q el más similar dentro de la matriz "descriptores"
# 3.imprimir documentos más similares